# Organisations Exploratory Data Analysis

## Objective

Check whether the organisation record is complete, whether its identifier is reliable, and whether it connects correctly to branches, departments and programmes.

## Files used

- `Organisations.csv` — organisation details
- `Branches.csv` — branches belonging to an organisation
- `Departments.csv` — departments belonging to an organisation
- `Programmes.csv` — wellness programmes belonging to an organisation and branch

The source contains only one organisation. This notebook can test the current record and its joins, but it cannot prove how the data will behave across many organisations.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

RAW_DATA_DIR = Path.cwd().parents[2] / "data" / "raw"
print("Raw data folder:", RAW_DATA_DIR)
print("Folder exists:", RAW_DATA_DIR.exists())

Raw data folder: /workspace/scratch/8819c521c579/verify_repo/data-analytics/data/raw
Folder exists: True


## 1. Load the data

In [2]:
organisations = pd.read_csv(RAW_DATA_DIR / "Organisations.csv")
branches = pd.read_csv(RAW_DATA_DIR / "Branches.csv")
departments = pd.read_csv(RAW_DATA_DIR / "Departments.csv")
programmes = pd.read_csv(RAW_DATA_DIR / "Programmes.csv")

dataset_summary = pd.DataFrame({
    "dataset": ["Organisations", "Branches", "Departments", "Programmes"],
    "rows": [len(organisations), len(branches), len(departments), len(programmes)],
    "columns": [len(organisations.columns), len(branches.columns), len(departments.columns), len(programmes.columns)],
})
dataset_summary

         dataset  rows  columns
0  Organisations     1       10
1       Branches     2        6
2    Departments     4        4
3     Programmes     1       11

## 2. Inspect the organisation record

In [3]:
organisations

  organisation_id                 name industry   country  employee_count  \
0         ORG-001  Kopano Mining Group   Mining  Botswana              28   

   status       plan   contract_start_date     contract_end_date  \
0  active  corporate  2026-06-01T00:00:00Z  2027-05-31T00:00:00Z   

             created_at  
0  2026-05-15T09:00:00Z  

In [4]:
organisation_profile = pd.DataFrame({
    "data_type": organisations.dtypes.astype(str),
    "missing_count": organisations.isna().sum(),
    "unique_values": organisations.nunique(dropna=False),
})
organisation_profile

                    data_type  missing_count  unique_values
organisation_id        object              0              1
name                   object              0              1
industry               object              0              1
country                object              0              1
employee_count          int64              0              1
status                 object              0              1
plan                   object              0              1
contract_start_date    object              0              1
contract_end_date      object              0              1
created_at             object              0              1

## 3. Check the organisation identifier

In [5]:
invalid_organisation_ids = organisations.loc[
    ~organisations["organisation_id"].str.match(r"^ORG-[0-9]{3,}$", na=False),
    ["organisation_id", "name"],
]

identifier_checks = pd.Series({
    "missing_ids": organisations["organisation_id"].isna().sum(),
    "duplicate_ids": organisations["organisation_id"].duplicated().sum(),
    "invalid_id_formats": len(invalid_organisation_ids),
    "duplicate_names_case_insensitive": organisations["name"].str.strip().str.casefold().duplicated().sum(),
})
identifier_checks.to_frame("count")

                                  count
missing_ids                           0
duplicate_ids                         0
invalid_id_formats                    0
duplicate_names_case_insensitive      0

In [6]:
assert organisations["organisation_id"].notna().all()
assert organisations["organisation_id"].is_unique
assert invalid_organisation_ids.empty
print("Organisation identifier checks passed.")

Organisation identifier checks passed.


## 4. Check values and contract dates

In [7]:
date_columns = ["contract_start_date", "contract_end_date", "created_at"]
for column in date_columns:
    organisations[column] = pd.to_datetime(organisations[column], utc=True, errors="coerce")

value_checks = pd.Series({
    "invalid_dates": organisations[date_columns].isna().sum().sum(),
    "non_positive_employee_counts": organisations["employee_count"].le(0).sum(),
    "contracts_ending_before_start": organisations["contract_end_date"].lt(organisations["contract_start_date"]).sum(),
    "records_created_after_contract_start": organisations["created_at"].gt(organisations["contract_start_date"]).sum(),
})
value_checks.to_frame("count")

                                      count
invalid_dates                             0
non_positive_employee_counts              0
contracts_ending_before_start             0
records_created_after_contract_start      0

In [8]:
controlled_values = pd.DataFrame({
    "field": ["industry", "country", "status", "plan"],
    "values_found": [
        sorted(organisations["industry"].dropna().unique().tolist()),
        sorted(organisations["country"].dropna().unique().tolist()),
        sorted(organisations["status"].dropna().unique().tolist()),
        sorted(organisations["plan"].dropna().unique().tolist()),
    ],
})
controlled_values

      field values_found
0  industry     [Mining]
1   country   [Botswana]
2    status     [active]
3      plan  [corporate]

In [9]:
assert value_checks.eq(0).all()
print("Organisation value and contract-date checks passed.")

Organisation value and contract-date checks passed.


### What this proves

The current organisation has a positive employee count and sensible contract dates. The values are complete. Because there is only one organisation, this file cannot show whether industry, country, status and plan values remain consistent when more clients are added.

## 5. Validate joins to branches and departments

In [10]:
orphan_branch_organisation_ids = sorted(set(branches["organisation_id"]) - set(organisations["organisation_id"]))
orphan_department_organisation_ids = sorted(set(departments["organisation_id"]) - set(organisations["organisation_id"]))

print("Unknown organisation IDs in Branches.csv:", orphan_branch_organisation_ids)
print("Unknown organisation IDs in Departments.csv:", orphan_department_organisation_ids)

Unknown organisation IDs in Branches.csv: []
Unknown organisation IDs in Departments.csv: []


In [11]:
branch_join = branches.merge(
    organisations[["organisation_id", "name", "country"]].rename(
        columns={"name": "organisation_name", "country": "organisation_country"}
    ),
    on="organisation_id",
    how="left",
    validate="many_to_one",
    indicator="organisation_join_status",
)
branch_join["country_matches"] = branch_join["country"].eq(branch_join["organisation_country"])
branch_join

  branch_id organisation_id                   name      city   country  \
0    BR-001         ORG-001  Metsi Operations Site   Jwaneng  Botswana   
1    BR-002         ORG-001   Gaborone Head Office  Gaborone  Botswana   

             created_at    organisation_name organisation_country  \
0  2026-05-15T09:05:00Z  Kopano Mining Group             Botswana   
1  2026-05-15T09:06:00Z  Kopano Mining Group             Botswana   

  organisation_join_status  country_matches  
0                     both             True  
1                     both             True  

In [12]:
department_join = departments.merge(
    organisations[["organisation_id", "name"]].rename(columns={"name": "organisation_name"}),
    on="organisation_id",
    how="left",
    validate="many_to_one",
    indicator="organisation_join_status",
)
department_join

  department_id organisation_id             name            created_at  \
0       DEP-001         ORG-001       Operations  2026-05-15T09:10:00Z   
1       DEP-002         ORG-001      Engineering  2026-05-15T09:11:00Z   
2       DEP-003         ORG-001          Finance  2026-05-15T09:12:00Z   
3       DEP-004         ORG-001  Human Resources  2026-05-15T09:13:00Z   

     organisation_name organisation_join_status  
0  Kopano Mining Group                     both  
1  Kopano Mining Group                     both  
2  Kopano Mining Group                     both  
3  Kopano Mining Group                     both  

In [13]:
assert not orphan_branch_organisation_ids
assert not orphan_department_organisation_ids
assert branch_join["organisation_join_status"].eq("both").all()
assert branch_join["country_matches"].all()
assert department_join["organisation_join_status"].eq("both").all()
print("All branches and departments link to the organisation correctly.")

All branches and departments link to the organisation correctly.


## 6. Validate joins to programmes

In [14]:
orphan_programme_organisation_ids = sorted(set(programmes["organisation_id"]) - set(organisations["organisation_id"]))
orphan_programme_branch_ids = sorted(set(programmes["branch_id"]) - set(branches["branch_id"]))

print("Unknown organisation IDs in Programmes.csv:", orphan_programme_organisation_ids)
print("Unknown branch IDs in Programmes.csv:", orphan_programme_branch_ids)

Unknown organisation IDs in Programmes.csv: []
Unknown branch IDs in Programmes.csv: []


In [15]:
programme_join = programmes.merge(
    organisations[["organisation_id", "name", "employee_count"]].rename(columns={"name": "organisation_name"}),
    on="organisation_id",
    how="left",
    validate="many_to_one",
    indicator="organisation_join_status",
).merge(
    branches[["branch_id", "organisation_id", "name"]].rename(
        columns={"organisation_id": "branch_organisation_id", "name": "branch_name"}
    ),
    on="branch_id",
    how="left",
    validate="many_to_one",
    indicator="branch_join_status",
)
programme_join["branch_belongs_to_programme_organisation"] = programme_join["organisation_id"].eq(programme_join["branch_organisation_id"])
programme_join

  programme_id organisation_id branch_id                               name  \
0      PRG-001         ORG-001    BR-001  2026 Workforce Wellness Programme   

  programme_type            start_date              end_date  \
0      screening  2026-06-18T00:00:00Z  2026-06-18T00:00:00Z   

                                   venue     status  target_participants  \
0  Metsi Operations Site - Wellness Hall  completed                   28   

             created_at    organisation_name  employee_count  \
0  2026-05-22T10:00:00Z  Kopano Mining Group              28   

  organisation_join_status branch_organisation_id            branch_name  \
0                     both                ORG-001  Metsi Operations Site   

  branch_join_status  branch_belongs_to_programme_organisation  
0               both                                      True  

In [16]:
assert not orphan_programme_organisation_ids
assert not orphan_programme_branch_ids
assert programme_join["organisation_join_status"].eq("both").all()
assert programme_join["branch_join_status"].eq("both").all()
assert programme_join["branch_belongs_to_programme_organisation"].all()
assert len(programme_join) == len(programmes)
print("All programmes link to the correct organisation and branch without losing rows.")

All programmes link to the correct organisation and branch without losing rows.


## 7. Simple conclusion


- Organisation ID is present, unique and written in the expected format.
- Important information such as the name, country, employee count, plan and contract dates is filled in.
- The contract dates are accurate.
- Both branches, all four departments and the wellness programme connect to the organisation correctly.
- The programme also connects to a branch that belongs to the same organisation.
- No records were lost during the joins.

The file contains only one organisation. It proves that the current organisation works, but it does not prove that the structure will work perfectly when platform has many clients.

The next step is to test the same checks again when more organisations are available. It will show whether names are duplicated and whether industry, country, plan and status values stay consistent.

**Overall result:** the current organisation data is complete and ready to join to branches, departments and programmes. More organisation records are needed before judging how well it will work at a larger scale.